# NEURAL NETWORK FROM SCRATCH

In [6]:
import math
import random


class Neuron:
    def __init__(self, num_inputs, weights=None, bias=None):
        # If weights and bias are provided, use them; otherwise, initialize randomly ( 0 - 1 )
        if weights is not None and bias is not None:
            self.weights = weights
            self.bias = bias
        else:
            self.weights = [random.random() for _ in range(num_inputs)]
            self.bias = random.random()

    def activate(self, inputs):
        self.inputs = inputs
        z = sum(w * i for w, i in zip(self.weights, inputs)) + self.bias
        self.output = self.sigmoid(z)
        return self.output

    def sigmoid(self, x):
        return 1 / (1 + math.exp(-x))


In [7]:
class Layer:
    def __init__(self, num_neurons, num_inputs_per_neuron, layer_weights=None, layer_biases=None):
        self.neurons = []
        for i in range(num_neurons):
            if layer_weights and layer_biases:
                self.neurons.append(Neuron(num_inputs_per_neuron, layer_weights[i], layer_biases[i]))
            else:
                self.neurons.append(Neuron(num_inputs_per_neuron))

    def forward(self, inputs):
        return [neuron.activate(inputs) for neuron in self.neurons]

In [8]:
class NeuralNetwork:
    def __init__(self, filepath):
        self.layers = []
        self.lines, self.layer_sizes, self.num_layers = self.parse_file(filepath)

    def parse_file(self, filepath):
        with open(filepath, 'r') as f:
            lines = []
            for line in f:
                line = line.strip()
                if line:
                    lines.append(line)

        # Based on the file, first line: number of layers N
        num_layers = int(lines[0])

        # Next N lines: number of neurons in each layer
        layer_sizes = []
        for i in range(1, num_layers + 1):
            layer_sizes.append(int(lines[i]))
        
        return lines, layer_sizes, num_layers
    
    def build_layers(self):
        current_line = self.num_layers + 1

        # Build layers based on the parsed architecture
        for i in range(1, len(self.layer_sizes)):
            num_inputs_per_neuron = self.layer_sizes[i-1]
            num_neurons = self.layer_sizes[i]
            layer_weights = []
            layer_biases = []
        
            for _ in range(num_neurons):
                # Parse weights and biases from the file
                values = list(map(float, self.lines[current_line].split()))
                layer_weights.append(values[:-1]) # All but the last item are weights
                layer_biases.append(values[-1])   # The last item is the bias
                current_line += 1

            self.layers.append(Layer(num_neurons, num_inputs_per_neuron, layer_weights, layer_biases))
    
    def forward(self, inputs):
        current_inputs = inputs
        for layer in self.layers:
            current_inputs = layer.forward(current_inputs)
        return current_inputs
    


xor_nn = NeuralNetwork("input.txt")
xor_nn.build_layers()

test_inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]

print("\nResults:")
for inputs in test_inputs:
    output = xor_nn.forward(inputs)

    # Predicted: Round the output to get the predicted class (0 or 1)
    predicted_class = round(output[0]) 
    print(f"Input: {inputs} -> Raw Output: {output[0]:.4f} ~ Predicted: {predicted_class}")


Results:
Input: [0, 0] -> Raw Output: 0.0072 ~ Predicted: 0
Input: [0, 1] -> Raw Output: 0.9924 ~ Predicted: 1
Input: [1, 0] -> Raw Output: 0.9924 ~ Predicted: 1
Input: [1, 1] -> Raw Output: 0.0072 ~ Predicted: 0
